# 02 — Pandas Data Cleaning

Clean data types, missing values, and duplicates.

In [3]:
from pathlib import Path

import pandas as pd
import numpy as np
import re

# ============================================================
# LOAD DATA
# ============================================================

notebook_dir = Path.cwd()
file_candidates = [
    notebook_dir / "shopsy_kitchen_products.csv",
    notebook_dir / "notebooks/shopsy_kitchen_products.csv",
    notebook_dir.parent / "notebooks/shopsy_kitchen_products.csv",
    Path("notebooks/shopsy_kitchen_products.csv")
]
file_path = next((path for path in file_candidates if path.exists()), None)

if file_path is None:
    raise FileNotFoundError("Raw CSV not found in the notebooks folder.")

df = pd.read_csv(file_path)

print("Before Cleaning:", df.shape)
display(df.head())

Before Cleaning: (30, 9)


,Product_Name,Price,Discount,Rating,Reviews,Brand,Capacity,Material,Product_URL
0,VASOYA Pack of 1 Plastic Fridge Container - 27...,₹161,86%,4.0,25.0,Shopsy,"2700 ml, 1500 ml, 500 ml",Plastic,https://www.shopsy.in/vasoya-pack-1-plastic-fr...
1,AneriDEALS Pack of 24 Plastic Grocery Containe...,₹423,78%,4.1,34.0,Shopsy,"250 ml, 350 ml, 650 ml, 1200 ml, 1000 ml",Plastic,https://www.shopsy.in/anerideals-pack-24-plast...
2,BELIZZI Pack of 6 Plastic Fridge Container - 1...,₹257,74%,4.1,272.0,Shopsy,"1500 ml, 500 ml, 1000 ml",Plastic,https://www.shopsy.in/belizzi-pack-6-plastic-f...
3,KIKANII Pack of 12 Plastic Grocery Container -...,₹351,64%,3.4,NaN,Shopsy,500 ml,Plastic,https://www.shopsy.in/kikanii-pack-12-plastic-...
4,COSY HOME MAKE IT EASY Pack of 1 Plastic Egg C...,₹158,80%,3.8,12.0,Shopsy,NaN,"Plastic, Silicone, Wood",https://www.shopsy.in/cosy-home-make-easy-pack...


In [8]:
# ============================================================
# 1. CLEAN PRODUCT NAME
# ============================================================

df["Product_Name"] = (
    df["Product_Name"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [6]:
# ============================================================
# 2. CLEAN PRICE
# ============================================================

df["Price"] = (
    df["Price"]
    .astype(str)
    .str.replace(r"[₹,\s]", "", regex=True)
)

df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

In [7]:
# ============================================================
# 3. CLEAN DISCOUNT
# ============================================================

df["Discount"] = (
    df["Discount"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.strip()
)

df["Discount"] = pd.to_numeric(df["Discount"], errors="coerce")

In [9]:
# ============================================================
# 4. CLEAN RATING
# ============================================================

df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df.loc[~df["Rating"].between(0, 5), "Rating"] = np.nan

In [10]:
# ============================================================
# 5. CLEAN REVIEWS
# ============================================================

df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce")
df["Reviews"] = df["Reviews"].round().astype("Int64")

In [11]:
# ============================================================
# 6. FIX BRAND
# ============================================================

def extract_brand(product_name):
    if pd.isna(product_name):
        return ""

    product_name = str(product_name).strip()
    match = re.match(
        r"^(.*?)\s+Pack\s+of\b",
        product_name,
        flags=re.I
    )

    if match:
        return match.group(1).strip()

    return ""


df["Brand"] = df["Product_Name"].apply(extract_brand)

In [12]:
# ============================================================
# 7. CLEAN CAPACITY
# ============================================================

df["Capacity"] = (
    df["Capacity"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


def clean_capacity(value):
    if not value:
        return ""

    parts = re.split(r",\s*", value)
    cleaned = []

    for part in parts:
        part = part.strip()
        if part and part.lower() not in [x.lower() for x in cleaned]:
            cleaned.append(part)

    return ", ".join(cleaned)


df["Capacity"] = df["Capacity"].apply(clean_capacity)

In [13]:
# ============================================================
# 8. CLEAN MATERIAL
# ============================================================

df["Material"] = (
    df["Material"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [14]:
# ============================================================
# 9. CLEAN PRODUCT URL
# ============================================================

df["Product_URL"] = (
    df["Product_URL"]
    .astype(str)
    .str.strip()
)

In [15]:
# ============================================================
# 10. REMOVE DUPLICATE PRODUCTS
# ============================================================

df = df.drop_duplicates(subset=["Product_URL"])

In [16]:
# ============================================================
# 11. HANDLE EMPTY VALUES
# ============================================================

df = df.replace(
    ["", "nan", "NaN", "None"],
    np.nan
)

In [17]:
# ============================================================
# 12. FINAL COLUMN ORDER
# ============================================================

df = df[
    [
        "Product_Name",
        "Price",
        "Discount",
        "Rating",
        "Reviews",
        "Brand",
        "Capacity",
        "Material",
        "Product_URL"
    ]
]

In [ ]:
# ============================================================
# 13. SAVE CLEAN CSV
# ============================================================

project_dir = notebook_dir if (notebook_dir / "Data").exists() else notebook_dir.parent
output_dir = project_dir / "Data" / "shopsy_db"
output_dir.mkdir(parents=True, exist_ok=True)
clean_file = output_dir / "shopsy_kitchen_products_cleaned.csv"

df.to_csv(
    clean_file,
    index=False,
    encoding="utf-8-sig"
)

print("Cleaned CSV saved successfully!")
print(clean_file)

OSError: Cannot save file into a non-existent directory: 'D:\DS\shopsy_home_kitchen_project\notebooks'

In [ ]:
# ============================================================
# 14. RESULTS
# ============================================================

print("\nAfter Cleaning:", df.shape)
print("\nMissing Values:")
print(df.isnull().sum())

print("\nCleaned Data:")
display(df.head(10))

print("\nSaved:")
print(clean_file)

In [16]:
print("Rows:", len(df))
print("Average price:", round(df["Price"].mean(), 2))
print("Average rating:", round(df["Rating"].mean(), 2))

Rows: 30
Average price: 295.97
Average rating: 4.16
